# JetNet Dataset: exploration

Code from the JetNet tutorial (maybe we can add/modify something)

## Data loading

Install the library and mount google drive to save the data

In [ ]:
!pip install jetnet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Import the JetNet dataset: let's see all available features for the jets and their particle content.

In [ ]:
from jetnet.datasets import JetNet

print(f"Particle features: {JetNet.ALL_PARTICLE_FEATURES}")
print(f"Jet features: {JetNet.ALL_JET_FEATURES}")

Download the data using the .getData method, which takes the data_args dictionary.

In [ ]:
data_args = {
    "jet_type": ["g", "t", "w"],  # gluon, top quark, and W boson jets
    "data_dir": "/content/drive/MyDrive/JetTagging/data",
    "particle_features": ["etarel", "phirel", "ptrel"], # features of the particles in the jet
    "num_particles": 30,
    "jet_features": ["type", "pt", "eta", "mass"],
    "download": True,
}

particle_data, jet_data = JetNet.getData(**data_args)

print(
    f"Particle features of the 10 highest pT particles in the first jet\n{data_args['particle_features']}\n{particle_data[0, :10]}"
)
print(f"\nJet features of first jet\n{data_args['jet_features']}\n{jet_data[0]}")

## Images visualization

The .to_image method allows a simple conversion to the image format, in which jets are represented on the (pseudorapidity, aximuthal angle) kinematic plane. On the z axis the transverse momentum p_T of the particle value is shown.

In [ ]:
from jetnet.utils import to_image
import matplotlib.pyplot as plt

num_images = 5
num_types = len(data_args["jet_type"])
im_size = 25  # number of pixels in height and width
maxR = 0.4  # max radius in (eta, phi) away from the jet axis

cm = plt.cm.jet.copy()
cm.set_under(color = "white")
plt.rcParams.update({"font.size": 16})

fig, axes = plt.subplots(
    nrows = num_types,
    ncols = num_images,
    figsize = (40, 8 * num_types),
    gridspec_kw = {"wspace": 0.25},
)

# get the index of each jet type using the JetNet.JET_TYPES array
type_indices = {jet_type: JetNet.JET_TYPES.index(jet_type) for jet_type in data_args["jet_type"]}

for j in range(num_types):
    jet_type = data_args["jet_type"][j]
    type_selector = jet_data[:, 0] == type_indices[jet_type]  # select jets based on jet_type feat

    axes[j][0].annotate(
        jet_type,
        xy = (0, -1),
        xytext = (-axes[j][0].yaxis.labelpad - 15, 0),
        xycoords = axes[j][0].yaxis.label,
        textcoords = "offset points",
        ha = "right",
        va = "center",
        fontsize = 24,
    )

    for i in range(num_images):
        im = axes[j][i].imshow(
            to_image(particle_data[type_selector][i], im_size, maxR = maxR),
            cmap = cm,
            interpolation = "nearest",
            vmin = 1e-8,
            extent = [-maxR, maxR, -maxR, maxR],
            vmax = 0.05,
        )
        axes[j][i].tick_params(which="both", bottom=False, top=False, left=False, right=False)
        axes[j][i].set_xlabel("$\phi^{rel}$")
        axes[j][i].set_ylabel("$\eta^{rel}$")
        axes[j][i].set_title(f"Jet {i + 1}")

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.01)
cbar.set_label("$p_T^{rel}$")